In [ ]:
#!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.8 MB/s eta 0:00:00


Blok ini menginstal library Faker, yang digunakan untuk menghasilkan data palsu yang terlihat realistis. Ini adalah praktik umum untuk membuat kumpulan data sintetis untuk pengujian atau demonstrasi.

1. import lbrary dan inisialisasi

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
import random

Blok ini mengimpor semua library Python yang diperlukan untuk manipulasi dan pembuatan data. numpy untuk operasi numerik, pandas untuk struktur data, Faker untuk pembuatan data sintetis, dan random untuk pembuatan angka acak umum.

2. membuat dataset sintetis

In [ ]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional
    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga, "quantity": qty,
        "date": tanggal,
        "payment_method": metode,
        "transaction_date":tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan
    # Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))


Jumlah baris: 515


Bagian ini membuat kumpulan data sintetis dari catatan transaksi. Ini menggunakan library Faker untuk menghasilkan nama pelanggan, detail produk, dan informasi terkait transaksi lainnya. Ini juga secara sengaja memperkenalkan nilai yang hilang dan baris duplikat untuk mensimulasikan ketidaksempurnaan data dunia nyata, yang akan ditangani pada langkah selanjutnya.

3. deteksi dan penanganan missing value

In [ ]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
date                  0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


Blok ini mendeteksi dan menampilkan jumlah total nilai yang hilang di setiap kolom DataFrame. Ini adalah langkah awal yang penting untuk memahami sejauh mana masalah kualitas data yang perlu ditangani.

In [ ]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah dropna():", len(df))

Jumlah baris setelah dropna(): 495


Blok ini menangani nilai-nilai yang hilang yang terdeteksi. Ini menghapus baris di mana ustomer_name atau payment_method hilang, karena ini adalah informasi penting. Untuk shipping_city, yang mungkin kurang penting, nilai yang hilang diisi dengan 'Tidak Diketahui' daripada menghapus seluruh baris.

4. deteksi penanganan duplicate

In [ ]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


Blok ini mengidentifikasi dan menghapus entri duplikat dari kumpulan data. Pertama, ia memeriksa baris yang merupakan duplikat persis di semua kolom, lalu menghapusnya, memastikan setiap catatan transaksi unik.

5. koreksi tiap data dan standarisasi format

a. standarisasi teks kategorikal

In [ ]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

Blok ini menstandarisasi data teks kategorikal. Ini membersihkan kolom seperti category, payment_method, dan shipping_city dengan menghapus spasi awal/akhir dan mengubahnya ke format judul, memastikan konsistensi untuk analisis. Koreksi khusus dibuat untuk Cod menjadi COD untuk mempertahankan kapitalisasi penuh.

b. koreksi tiap data pada kolom price

In [ ]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

Blok ini membersihkan dan menstandarisasi kolom price. Ini mendefinisikan fungsi untuk menghapus karakter non-numerik seperti Rp dan titik, lalu mengubah string yang dibersihkan menjadi angka floating-point, menangani berbagai format input dan potensi kesalahan dengan baik.

c. standarisasi format tanggal ke YYYY-MM-DD

In [ ]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

Blok ini menstandarisasi kolom transaction_date. Ini mendefinisikan fungsi yang mencoba mengurai tanggal dari berbagai format (YYYY-MM-DD, DD/MM/YYYY, DD-MM-YYYY) dan mengubahnya menjadi format string 'YYYY-MM-DD' yang konsisten, sangat penting untuk analisis deret waktu.

d. finalisasi tipe data:

In [ ]:
df["quantity"] = df["quantity"].astype("int")
df["price"] = df["price"].astype("float")

Blok ini melakukan konversi tipe data akhir untuk kolom tertentu. Ini memastikan bahwa quantity disimpan sebagai integer dan price sebagai float, yang merupakan tipe data yang sesuai untuk perhitungan numerik dan konsisten dengan upaya pembersihan data.

6. ekspor dataset bersih

In [ ]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


Blok terakhir ini mengekspor DataFrame yang telah dibersihkan dan diproses secara menyeluruh ke dalam file CSV baru bernama transaksi_bersih.csv. Ini menandakan selesainya pipeline pra-pemrosesan data, membuat data siap untuk analisis atau pemodelan lebih lanjut.

# analisis
Dari 515 baris data mentah, 25 baris (sekitar 4,9%) dibuang selama proses pra-pemrosesan: 15 baris
karena duplicate dan 10 baris karena kehilangan data wajib (customer_name atau payment_method).
Ini adalah gambaran realistis, kehilangan data selama proses cleaning adalah hal normal, bukan tanda
ada yang salah justru tanda bahwa proses cleaning bekerja dengan benar.
Kolom rating sengaja tidak diisi (imputed) karena mengisi rating yang kosong dengan angka tebakan
akan mendistorsi analisis di kemudian hari, lebih baik membiarkannya kosong dan menanganinya secara
eksplisit saat dianalisis nanti.
Aturan penulisan analisis: satu klaim, satu angka, satu alasan. Hindari kalimat tanpa bukti seperti “data
jadi lebih baik karena sudah dibersihkan”.

#latihan
1. Latihan 1: Ubah SEED menjadi 7 dan jalankan ulang seluruh pipeline. Bandingkan jumlah baris
transaksi_mentah.csv dan transaksi_bersih.csv dengan hasil SEED = 42. Apakah
jumlahnya sama? Jelaskan mengapa.
2. Latihan 2: Tambahkan kolom is_valid_price bernilai True jika price > 0. Gunakan untuk
memeriksa apakah ada harga tidak valid.
3. Latihan 3: Hitung jumlah transaksi per category menggunakan value_counts() pada dataset yang
sudah bersih.

**jawab**
1. ya, karean rumus pembuat data dan logika pembersihan dalam kode bersifat deterministik berdasarkan proporsi fixed (seperti duplikasi 15 baris dan hapus duplikat), sehingga mengubah nilai SEED hanya mengganti variasi isi datanya, bukan jumlah barisnya.

In [ ]:
#2.
df["is_valid_price"] = df["price"] > 0
print(df["is_valid_price"].value_counts())

is_valid_price
True    490
Name: count, dtype: int64


In [ ]:
#3.
print("\nJumlah Transaksi per Kategori:")
print(df["category"].value_counts())


Jumlah Transaksi per Kategori:
category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64


In [ ]:
#export file bersih:
df.to_csv("transaksi_bersih.csv", index=False)

# studi kasus

1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda
lakukan.
2. Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep
Veracity.
3. Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing
value kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?

**jawab**
1. Angka 515 dari tim IT adalah jumlah data mentah (raw data) yang baru dikumpulkan, yang di dalamnya masih terdapat transaksi duplikat akibat kendala teknis (seperti gangguan jaringan saat pembayaran). Sedangkan angka 490 milik tim Finance adalah jumlah data bersih setelah dilakukan proses preprocessing untuk mendeteksi dan menghapus 25 baris transaksi duplikat tersebut.
2. ya, angka 490 baris "lebih benar" dan valid. Berdasarkan konsep Veracity (keakuratan dan keandalan data), data mentah berjumlah 515 baris mengandung bias yang bisa menyebabkan laporan keuangan overstated (tercatat ganda). Mengacu pada prinsip garbage in, garbage out, pembersihan data duplikat sangat penting agar data yang dipakai tim Finance benar-benar mencerminkan transaksi nyata.
3. Kolom rating bersifat opsional karena pembeli tidak diwajibkan memberikan ulasan setelah bertransaksi. Membiarkan missing value tetap kosong (tidak diisi angka 0 atau rata-rata) adalah langkah tepat agar perhitungan rating rata-rata tetap akurat. Jika diisi angka 0, nilai rata-rata rating akan turun secara tidak alami dan merusak keaslian data ulasan pelanggan.

# Tugas praktikum
1. Seluruh pipeline Langkah K-1 s.d. K-6 berjalan tanpa error, dari atas ke bawah.
2. Jawaban tertulis (dalam sel Markdown di notebook) untuk ketiga pertanyaan Studi Kasus.
3. Hasil ketiga nomor Latihan, lengkap dengan output-nya.
4. File transaksi_bersih.csv yang dihasilkan, dilampirkan bersama notebook.